# AuraSDK — Quickstart (zero install, runs in your browser)

[Aura Memory](https://github.com/teolex2020/aura-memory) is a **cognitive memory layer for AI agents**. It learns from interactions — extracting preferences, causal patterns, and behavioral policies — **without any LLM calls, embeddings, cloud, or fine-tuning**.

This notebook runs the 60-second demo end to end. Just press **Runtime → Run all**.

- 5-layer cognitive hierarchy: `Record → Belief → Concept → Causal → Policy`
- Fully local & deterministic
- ~3 MB native binary, no heavy dependencies

## 1. Install

In [ ]:
%pip install -q aura-memory

## 2. Create a brain and store interactions

Memories live at different cognitive levels. `Level.Domain` holds stable knowledge; `Level.Decisions` holds decisions and their outcomes.

In [ ]:
import tempfile
from aura import Aura, Level

brain = Aura(tempfile.mkdtemp(prefix="aura_demo_"))
brain.enable_full_cognitive_stack()

# User preferences (stable knowledge)
preferences = [
    ("User always reviews code before merging to main branch", ["workflow", "code-review"]),
    ("User prefers detailed error messages over silent failures", ["workflow", "errors"]),
    ("User wants notifications only for critical issues", ["notifications", "workflow"]),
    ("User likes concise answers, not long explanations", ["communication"]),
    ("User prefers dark mode in all tools", ["ui", "preferences"]),
    ("User schedules deep work in morning, meetings in afternoon", ["schedule", "productivity"]),
    ("User prefers async communication over meetings when possible", ["communication", "workflow"]),
]
for content, tags in preferences:
    brain.store(content, level=Level.Domain, tags=tags)

print(f"Stored {len(preferences)} preferences")

In [ ]:
# Decisions and their outcomes
events = [
    ("Deployed directly to production -- caused 2h outage", ["deploy", "incident"], "deploy-1"),
    ("Added staging environment -- caught 3 bugs before prod", ["deploy", "staging"], "deploy-2"),
    ("Staging deploy prevented database migration failure", ["deploy", "staging", "database"], "deploy-3"),
    ("Direct prod deploy skipped staging -- caused data loss", ["deploy", "incident"], "deploy-4"),
    ("Staging review caught API breaking change in time", ["deploy", "staging", "api"], "deploy-5"),
    ("Skipped code review -- introduced security vulnerability", ["code-review", "security"], "review-1"),
    ("Code review caught SQL injection before merge", ["code-review", "security"], "review-2"),
    ("Code review found performance regression early", ["code-review", "performance"], "review-3"),
    ("Merged without review -- broke production auth", ["code-review", "incident"], "review-4"),
    ("Critical alert ignored -- escalated to incident", ["notifications", "incident"], "notif-1"),
    ("Non-critical noise alerts caused alert fatigue", ["notifications", "workflow"], "notif-2"),
    ("Filtered notifications to critical-only -- response time improved", ["notifications", "workflow"], "notif-3"),
]
ids = {}
for content, tags, key in events:
    result = brain.store(content, level=Level.Decisions, tags=tags)
    ids[key] = result if isinstance(result, str) else result.id

print(f"Stored {len(events)} decisions with outcomes")

## 3. Run the learning cycle

No LLM calls, no cloud, no fine-tuning — pure local computation.

In [ ]:
import time

t0 = time.perf_counter()
for _ in range(5):
    brain.run_maintenance()
print(f"5 learning cycles complete in {(time.perf_counter() - t0) * 1000:.1f} ms")

## 4. See what the system learned on its own

In [ ]:
concepts = brain.get_surfaced_concepts(5)
if concepts:
    print("Discovered topic clusters:")
    for c in concepts:
        label = getattr(c, "label", None) or getattr(c, "key", str(c))
        print(f"  * {label}  ({len(c.record_ids)} observations, score={c.score:.2f})")
else:
    print("Topic clusters: forming (need more cycles on real data)")

hints = brain.get_surfaced_policy_hints(5)
if hints:
    print("\nDerived behavioral policies (no one wrote these rules):")
    for h in hints:
        action = getattr(h, "action_kind", getattr(h, "action", "Recommend"))
        desc = getattr(h, "description", str(h))
        strength = getattr(h, "strength", 0.0)
        print(f"  [{action}] {desc[:70]}  (confidence={strength:.2f})")
else:
    print("\nBehavioral policies: accumulating (run more cycles on real data)")

## 5. Recall — what gets injected into your LLM prompt

In [ ]:
for q in ["deployment decision", "code review", "user preferences"]:
    t0 = time.perf_counter()
    results = brain.recall_structured(q, top_k=3, min_strength=0.0)
    elapsed = (time.perf_counter() - t0) * 1000
    print(f'\nQuery: "{q}"  [{elapsed:.2f} ms]')
    for i, r in enumerate(results[:2], 1):
        content = r["content"] if isinstance(r, dict) else r.content
        print(f"  {i}. {content[:70]}")

## What just happened

From a handful of interactions the cognitive layer extracted user-preference patterns, causal relationships between decisions and outcomes, and behavioral policies your agent can act on — with **zero LLM calls, zero cloud, zero fine-tuning**, fully offline.

Next steps:
- ⭐ Star the repo: https://github.com/teolex2020/aura-memory
- Browse runnable integrations (Ollama, LangChain, Claude SDK, CrewAI, FastAPI, …) in [`examples/`](https://github.com/teolex2020/aura-memory/tree/main/examples)
- Read the [README](https://github.com/teolex2020/aura-memory/blob/main/README.md) for the full cognitive architecture